# Lecture 1.1: Basic Elements in Neural Networks; 1.1 Perceptron and MLP

This notebook introduces fundamental concepts of Neural Networks using PyTorch. It covers:

*   **PyTorch Tensors**: Basic data structures for computations.
*   **Activation Functions**: Essential non-linearities (Sigmoid, ReLU, Tanh, GELU, Leaky ReLU, SiLU, Heaviside, Softplus, Hardtanh) and their visualization.
*   **Perceptron Implementations**: From manual Python classes to `nn.Module` with `nn.Linear`.
*   **Multi-Layer Perceptrons (MLP)**: Building simple feed-forward networks using `nn.Sequential`.
*   **`nn.Embedding`**: An exercise demonstrating its use and parameter counting in an Item Classifier model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_1_1_perceptron_mlp_pytorch.ipynb)

*(Remember to update the GitHub path in the Colab badge above to your actual repository path.)*

## PyTorch Fundamentals: Tensors and Activation Functions

### What are Tensors?

In PyTorch, a **Tensor** is the fundamental data structure, very similar to a NumPy array. Tensors are specialized data structures that are very similar to arrays and matrices. In PyTorch, we use tensors to encode the inputs and outputs of a model, as well as the model’s parameters. They are designed to work efficiently on GPUs, which is crucial for deep learning.

Let's create a simple tensor.

In [ ]:
import torch

# Create a simple tensor from a Python list
x = torch.tensor([[1., 2.], [3., 4.]])
print(f"Tensor: {x}")
print(f"Shape: {x.shape}")
print(f"Data type: {x.dtype}")

# Create a tensor of zeros
zeros_tensor = torch.zeros(2, 3)
print(f"\nZeros Tensor: {zeros_tensor}")

# Create a tensor of ones
ones_tensor = torch.ones(3, 2)
print(f"\nOnes Tensor: {ones_tensor}")

# Create a random tensor
rand_tensor = torch.rand(4)
print(f"\nRandom Tensor: {rand_tensor}")


### Activation Functions

Activation functions are crucial components in neural networks. They introduce non-linearity into the model, allowing it to learn complex patterns in data. Without them, a neural network would just be a series of linear operations, limiting its ability to solve non-linear problems.

Let's visualize some common activation functions.

In [ ]:
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Prepare a range of input values for plotting
x_np = np.linspace(-5, 5, 100)
x_torch = torch.tensor(x_np, dtype=torch.float32)

def plot_activation(ax, y, title):
    ax.plot(x_np, y)
    ax.set_title(title)
    ax.grid(True)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()
plot_idx = 0

y_heaviside = torch.heaviside(x_torch, torch.tensor(0.5)) # 0.5 is the value at x=0
plot_activation(axes[plot_idx], y_heaviside, 'Heaviside Step Function')
plot_idx += 1

y_sigmoid = torch.sigmoid(x_torch).numpy()
plot_activation(axes[plot_idx], y_sigmoid, 'Sigmoid')
plot_idx += 1

y_tanh = torch.tanh(x_torch).numpy()
plot_activation(axes[plot_idx], y_tanh, 'Tanh')
plot_idx += 1

y_relu = F.relu(x_torch).numpy()
plot_activation(axes[plot_idx], y_relu, 'ReLU')
plot_idx += 1

y_leaky_relu = F.leaky_relu(x_torch).numpy()
plot_activation(axes[plot_idx], y_leaky_relu, 'Leaky ReLU')
plot_idx += 1

y_silu = F.silu(x_torch).numpy()
plot_activation(axes[plot_idx], y_silu, 'SiLU')
plot_idx += 1

y_gelu = F.gelu(x_torch).numpy()
plot_activation(axes[plot_idx], y_gelu, 'GELU')
plot_idx += 1

# Softplus: Smooth approximation to the ReLU function
y_softplus = F.softplus(x_torch).numpy()
plot_activation(axes[plot_idx], y_softplus, 'Softplus')
plot_idx += 1

# Hardtanh: Clamps values between min_val and max_val
y_hardtanh = F.hardtanh(x_torch, min_val=-1.0, max_val=1.0).numpy() # Default range is [-1, 1]
plot_activation(axes[plot_idx], y_hardtanh, 'Hardtanh')
plot_idx += 1

plt.tight_layout()
plt.show()

#### A Note on PyTorch Activation Functions: `F.functional` vs. `nn.Module`

In PyTorch, you often see two ways to use activation functions (and other operations):

1.  **`torch.nn.functional` (e.g., `F.sigmoid`, `F.relu`):**
    *   These are **pure functions** that simply take a tensor as input and return a transformed tensor.
    *   They are **stateless**, meaning they don't "remember" anything or have any internal configuration. You just call them directly when you need them.
    *   We use these in the plots above because they are straightforward for one-off calculations.
    *   Note that you can also use some some of them as e.g. `torch.sigmoid`. This is equivalent to `F.sigmoid`

2.  **`torch.nn` (e.g., `nn.Sigmoid()`, `nn.ReLU()`):**
    *   These are **classes** that inherit from `nn.Module` (PyTorch's base class for all neural network building blocks).
    *   You create an **object** from these classes (e.g., `my_sigmoid = nn.Sigmoid()`).
    *   They are designed to be **layers** within a neural network model. They can hold internal *state* (like learnable parameters in more complex layers, though not in simple Sigmoid).
    *   They integrate better into larger neural network structures, allowing PyTorch to manage them as part of the model (e.g., counting parameters, moving the entire model to a GPU).

**In short:** Use the `F.` versions for quick, functional operations. Use the `nn.` versions when you want to define a formal "layer" that is part of a larger neural network structure.



In [ ]:
import torch.nn as nn

# Create a figure with two subplots
fig_compare, axes_compare = plt.subplots(1, 2, figsize=(12, 5))

# Calculate F.sigmoid
y_f_sigmoid = F.sigmoid(x_torch).numpy()
plot_activation(axes_compare[0], y_f_sigmoid, 'F.sigmoid() (Functional)')

# Instantiate nn.Sigmoid and calculate its output
nn_sigmoid_layer = nn.Sigmoid()
y_nn_sigmoid = nn_sigmoid_layer(x_torch).numpy()
plot_activation(axes_compare[1], y_nn_sigmoid, 'nn.Sigmoid() (Module)')

plt.tight_layout()
plt.show()

### The Perceptron: Understanding the Matrix Product

At the heart of every neuron in a neural network is a simple mathematical operation: a **matrix product** (also called a dot product for vectors) followed by an addition, and then an activation function. This is often written as `z = Wx + b`.

*   `x`: This is your input vector (e.g., `[X1, X2]`).
*   `W`: This is the weight matrix (or vector, for a single neuron). It defines how important each input feature is to the neuron's output.
*   `b`: This is the bias term. It allows the neuron to shift its activation independently of its input.
*   `z`: This is the "pre-activation" or "net input." It's the result of the weighted sum of inputs plus the bias.
*   `h = activation(z)`: The activation function then transforms `z` into the neuron's final output, introducing non-linearity.

Let's demonstrate this fundamental operation with a simple example before building different perceptron versions.

In [ ]:
# Example of a simple matrix product (dot product for a single neuron)

# Input vector (e.g., a single data point for X1, X2)
x_input = torch.tensor([0.5, 0.8], dtype=torch.float32)
print(f"Input x: {x_input}")

# Weight vector for a single neuron (in_features=2, out_features=1)
weights = torch.tensor([0.7, -0.3], dtype=torch.float32)
print(f"Weights W: {weights}")

# Bias term for the neuron
bias = torch.tensor([0.1], dtype=torch.float32)
print(f"Bias b: {bias}")

# Calculate the pre-activation 'z' using matrix multiplication
# For a single output neuron, this is a dot product
# torch.matmul(input, weights) or input @ weights performs xW^T for us implicitly if shapes allow
z = torch.matmul(x_input, weights) + bias # Or simply x_input @ weights + bias
print(f"Pre-activation z = (x @ W) + b: {z}")

# Apply a sigmoid activation
h = torch.sigmoid(z)
print(f"Activated output h = sigmoid(z): {h}")

### Implementing a Perceptron in Different Ways

Now, let's see how we can implement a single perceptron using different PyTorch and Python patterns. This will help you understand the abstraction layers.

#### Version 1: Custom Python Class with Manual Matrix Multiplication

This version uses a standard Python class to define a perceptron. We explicitly define weights and biases as PyTorch tensors and implement the `__call__` method to make the object callable like a function. This is the most 'from scratch' approach, demonstrating the raw mathematical operations.

In [ ]:
class PerceptronManual:
    def __init__(self, in_features):
        # Initialize weights and bias as PyTorch tensors
        # We make them require_grad=True so PyTorch can track them for learning
        self.weights = torch.randn(in_features, requires_grad=True)
        self.bias = torch.randn(1, requires_grad=True)

    def __call__(self, x):
        # This method makes the object callable, e.g., `my_perceptron(input_tensor)`
        # Perform the matrix multiplication (dot product for a single neuron)
        z = torch.matmul(x, self.weights) + self.bias
        # Apply the sigmoid activation
        h = torch.sigmoid(z)
        return h

print("--- PerceptronManual ---")
# Instantiate the custom perceptron for 2 input features
manual_perceptron = PerceptronManual(in_features=2)

# Example input
input_data = torch.tensor([0.5, 0.8], dtype=torch.float32)

# Get the output
output = manual_perceptron(input_data)

print(f"Input: {input_data}")
print(f"Weights: {manual_perceptron.weights.data.numpy()}")
print(f"Bias: {manual_perceptron.bias.data.numpy()}")
print(f"Output: {output.item():.4f}")

#### Version 2: Custom `nn.Module` with Manual `nn.Parameter` and Matrix Multiplication

This version introduces `torch.nn.Module`, which is the base class for all neural network modules in PyTorch. By inheriting from `nn.Module`, our perceptron can leverage PyTorch's powerful features like automatic parameter tracking, moving the model to different devices (CPU/GPU), and integrating into larger networks. We still explicitly define weights and biases, but now as `nn.Parameter` objects to signal to PyTorch that these are learnable parameters.

In [ ]:
class PerceptronCustomNN(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        # Initialize weights and bias using nn.Parameter
        # nn.Parameter automatically registers them as learnable parameters of the module
        self.weights = nn.Parameter(torch.randn(in_features))
        self.bias = nn.Parameter(torch.randn(1))

    def forward(self, x):
        # The forward method defines the computation performed at every call
        # Perform the matrix multiplication
        z = torch.matmul(x, self.weights) + self.bias
        # Apply the sigmoid activation
        h = torch.sigmoid(z)
        return h

print("--- PerceptronCustomNN ---")
# Instantiate the custom nn.Module perceptron
custom_nn_perceptron = PerceptronCustomNN(in_features=2)

# Example input
input_data = torch.tensor([0.5, 0.8], dtype=torch.float32)

# Get the output
output = custom_nn_perceptron(input_data) # Calling the module automatically calls its forward method

print(f"Input: {input_data}")
print(f"Weights: {custom_nn_perceptron.weights.data.numpy()}")
print(f"Bias: {custom_nn_perceptron.bias.data.numpy()}")
print(f"Output: {output.item():.4f}")

# We can also inspect its parameters, which is a key advantage of nn.Module
print("\nLearnable parameters:")
for name, param in custom_nn_perceptron.named_parameters():
    print(f"  {name}: {param.shape}")

#### Version 3: Custom `nn.Module` using `nn.Linear`

This is the most common and recommended way to define a perceptron (or any fully connected layer) in PyTorch. `nn.Linear` is a pre-built `nn.Module` that handles the creation and management of weights and biases, as well as the matrix multiplication (`Wx + b`) automatically. It abstracts away the manual `nn.Parameter` and `torch.matmul` steps, making your code cleaner and less error-prone.

We combine `nn.Linear` with an activation function (like `nn.Sigmoid()` or `torch.sigmoid()`) to complete our perceptron.

In [ ]:
class PerceptronLinearNN(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        # nn.Linear handles the weights and bias for the linear transformation
        self.linear_layer = nn.Linear(in_features=in_features, out_features=1) # out_features=1 for a single perceptron

    def forward(self, x):
        # Apply the linear transformation (Wx + b)
        z = self.linear_layer(x)
        # Apply the sigmoid activation
        h = torch.sigmoid(z)
        return h

print("--- PerceptronLinearNN ---")
# Instantiate the nn.Linear-based perceptron
linear_nn_perceptron = PerceptronLinearNN(in_features=2)

# Example input
input_data = torch.tensor([0.5, 0.8], dtype=torch.float32)

# Get the output
output = linear_nn_perceptron(input_data)

print(f"Input: {input_data}")

# Weights and bias are now managed within linear_layer
print(f"Weights (from nn.Linear): {linear_nn_perceptron.linear_layer.weight.data.numpy()}")
print(f"Bias (from nn.Linear): {linear_nn_perceptron.linear_layer.bias.data.numpy()}")
print(f"Output: {output.item():.4f}")

# Inspecting parameters is still easy
print("\nLearnable parameters:")
for name, param in linear_nn_perceptron.named_parameters():
    print(f"  {name}: {param.shape}")

### Multi-Layer Perceptron (MLP) with `nn.Sequential`

For building more complex neural networks, PyTorch provides `nn.Sequential`. This is a container module that allows you to easily stack multiple `nn.Module` layers in a sequential order. The output of one layer automatically becomes the input of the next. It's a very straightforward way to define feed-forward networks.

Let's implement the MLP structure you showed in your slide. I'll use a dummy input to demonstrate its `forward` pass.

In [ ]:
import torch
import torch.nn as nn

# Define the MLP model using nn.Sequential
# Note: Corrected 'nn.Sequencial' to 'nn.Sequential' and 'nn.linear' to 'nn.Linear'
model_mlp = nn.Sequential(
    nn.Linear(28*28, 128), # Input layer to first hidden layer
    nn.ReLU(),             # Activation function
    nn.Linear(128, 100),   # First hidden layer to second hidden layer
    nn.ReLU(),             # Activation function
    nn.Linear(100, 10),    # Second hidden layer to output layer
    nn.Softmax(dim=-1)     # Softmax activation for probabilities over 10 classes
)

print("--- MLP Model (using nn.Sequential) ---")
print(model_mlp)

# Calculate and print total parameters
total_mlp_params = sum(p.numel() for p in model_mlp.parameters())
print(f"Total MLP parameters: {total_mlp_params:,}")

# Create a dummy input tensor
# For an image of 28x28 pixels, we'll flatten it to 784 features.
# Let's assume a batch size of 1 for this example.
dummy_input = torch.randn(1, 28*28)
print(f"\nShape of dummy input: {dummy_input.shape}")

# Pass the dummy input through the model to get probabilities
probs = model_mlp(dummy_input)

print(f"Shape of output probabilities: {probs.shape}")
print(f"Output Probabilities (first 5 values): {probs[0, :5].detach().numpy()}")
print(f"Sum of output probabilities (should be close to 1): {probs.sum().item():.4f}")

### Exercise: e-commerce Item Classifier: Model Parameters

Let's consider an e-commerce platform that needs to classify products into 10 primary departments based on their unique Item ID. The proposed architecture is:

*   **Input**: Item ID (1 out of 10,000 total catalog products).
*   **Embedding Layer**: Projects the discrete Item ID to a 100-dimensional vector.
*   **Backbone**: 3 hidden linear layers with 100 units each, followed by ReLU activation.
*   **Output Layer**: A linear layer with 10 units, followed by Softmax (to predict the 10 departments).

**Question**: How many trainable parameters does this model have in total, and for each of its main components (Embedding, Backbone, Output Layer)?

#### Understanding `nn.Embedding`

Before we define the model, let's briefly understand `nn.Embedding`.

`torch.nn.Embedding` is a module that creates a lookup table for embeddings. It's typically used to represent discrete input categories (like words, users, or item IDs) as dense, continuous vectors. Conceptually, an embedding layer can be thought of as applying a **one-hot encoding** to your input IDs, followed by a **linear layer** that maps this high-dimensional one-hot vector to a lower-dimensional dense vector. However, `nn.Embedding` does this much more efficiently, directly looking up the vector for each ID rather than performing a large matrix multiplication.

*   `num_embeddings` (or `num_items` in our case): This is the size of the dictionary of embeddings, i.e., the total number of unique discrete items you have (e.g., 10,000 unique Item IDs).
*   `embedding_dim`: This is the size of each embedding vector, i.e., the dimension of the continuous vector that each Item ID will be mapped to (e.g., 100 dimensions).

In [ ]:
import torch
import torch.nn as nn

class ItemClassifier(nn.Module):
    def __init__(self, num_items, embedding_dim, hidden_dim, output_dim):
        super(ItemClassifier, self).__init__()

        # 1. Embedding Layer: Converts item IDs into a dense vector representation.
        # num_items: The total number of unique items (10000 in our case).
        # embedding_dim: The size of the vector for each item (100).
        self.embedding = nn.Embedding(num_items, embedding_dim)

        # 2. Backbone: A sequence of fully connected (linear) layers with ReLU activations.
        # This part processes the embedded item ID.
        self.backbone = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),  # First hidden layer
            nn.ReLU(),                              # Activation function
            nn.Linear(hidden_dim, hidden_dim),      # Second hidden layer
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),      # Third hidden layer
            nn.ReLU()
        )

        # 3. Output Layer: Maps the output of the backbone to the final number of departments.
        # This layer will predict the probabilities for 10 different departments.
        self.output_layer = nn.Linear(hidden_dim, output_dim)

        # Softmax is often applied after the linear layer in the loss function (e.g., CrossEntropyLoss),
        # but we can also add it explicitly here for demonstration if needed.
        # Using dim=-1 (last dimension) makes it work for both a single item ID (vector) or a batch of item IDs.
        self.softmax = nn.Softmax(dim=-1) # Apply softmax along the last dimension

    def forward(self, item_id):
        # Pass item_id through embedding layer
        embedded_item = self.embedding(item_id)
        # Pass embedded item through the backbone
        backbone_output = self.backbone(embedded_item)
        # Pass backbone output through the final linear layer
        logits = self.output_layer(backbone_output)
        # Apply softmax to get probabilities
        probabilities = self.softmax(logits)
        return probabilities

# Model parameters based on the exercise description
num_items = 10000  # Total catalog products
embedding_dim = 100 # Dimension of the item embedding
hidden_dim = 100    # Units in each hidden layer of the backbone
output_dim = 10     # Number of departments to predict

# Create an instance of our model
model = ItemClassifier(num_items, embedding_dim, hidden_dim, output_dim)

print("--- ItemClassifier Model ---")
print(model)

# Calculate and print total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal model parameters: {total_params:,}")

# Parameters for Embedding Layer
embedding_params = sum(p.numel() for p in model.embedding.parameters())
print(f"Embedding Layer Parameters: {embedding_params:,}")

# Parameters for Backbone (hidden layers)
backbone_params = sum(p.numel() for p in model.backbone.parameters())
print(f"Backbone (3 hidden linear layers) Parameters: {backbone_params:,}")

# Parameters for Output Layer
output_layer_params = sum(p.numel() for p in model.output_layer.parameters())
print(f"Output Layer Parameters: {output_layer_params:,}")

# Demonstrate with a dummy input
dummy_item_id = torch.tensor([500], dtype=torch.long) # A single item ID, ID=500
output_probabilities = model(dummy_item_id)
predicted_department = torch.argmax(output_probabilities, dim=-1).item()

print(f"\nOutput probabilities for item ID {dummy_item_id.item()}: {output_probabilities.shape}")
print(f"Predicted department (based on max probability): {predicted_department}")